# 🏷️ Мониторинг цен конкурентов на красную икру (v2)

**Накопительная система: таблица продуктов растёт с каждым запуском, а ты вручную указываешь, какие продукты сравнимы.**

Нажми `Runtime` → `Run all` (или `Среда выполнения` → `Выполнить всё`) — и всё запустится автоматически.

---

### Как это работает (v2)

1. **Первый запуск**: скрипт находит ВСЕ продукты красной икры на 3 сайтах и сохраняет в `data/products.csv`
2. **Ты открываешь CSV** в Excel / Google Sheets и в колонке `is_comparable` ставишь:
   - **«да»** — продукт похож на наш, отслеживаем его цену
   - **«нет»** — продукт не подходит, больше не обновляем
3. **Повторный запуск**: скрипт обновляет цены только для «да», пропускает «нет», добавляет новинки

### Колонки в таблице

| Колонка | Что это | Кто заполняет |
|---|---|---|
| `site` | Магазин | Скрипт |
| `product_name` | Название продукта | Скрипт |
| `weight_g` | Масса (грамм) | Скрипт |
| `original_price_rub` | Цена за упаковку | Скрипт |
| `comparable_price_rub` | Цена за наши 120 г | Скрипт |
| `in_stock` | В наличии (да/нет) | Скрипт |
| `check_status` | Проверен (да/нет) | Скрипт |
| **`is_comparable`** | **Сопоставим?** | **ТЫ ✍️** |
| `url` | Ссылка на продукт | Скрипт |

### Наш продукт

**Икра горбуши 120 г** — базовая цена 1 590 ₽ (измени число в Шаге 3)

## Шаг 1: Установка зависимостей

Устанавливаем библиотеки, нужные для сбора данных с сайтов.

In [ ]:
# ============================================================
# ШАГ 1: Установка Python-пакетов
# ============================================================

# httpx — библиотека для HTTP-запросов (как requests, но быстрее)
# Нужна чтобы скачивать страницы сайтов и вызывать API
!pip install -q httpx

# lxml — библиотека для разбора HTML
# Нужна чтобы вытаскивать названия и цены из HTML-кода страниц
!pip install -q lxml

# playwright — браузерная автоматизация
# Нужна для сайта delikateska.ru (запасной вариант, если API не ответит)
# -q значит "тихий режим" — меньше вывода в лог
!pip install -q playwright

# Устанавливаем браузер Chromium для Playwright
# Это загружает ~120 МБ один раз
!playwright install chromium 2>&1 | tail -3

print("✅ Зависимости установлены")

## Шаг 2: Загрузка проекта

Клонируем проект с GitHub — это наш код для сбора цен.

In [ ]:
# ============================================================
# ШАГ 2: Клонирование проекта с GitHub
# ============================================================

# git clone — команда для скачивания репозитория
# https://github.com/tswtim/price-monitor.git — адрес твоего репозитория
# Весь код (адаптеры, нормализация, отчёты) лежит в этом репозитории
!git clone -q https://github.com/tswtim/price-monitor.git

# %cd — магическая команда Jupyter: перейти в папку проекта
# Все дальнейшие команды будут выполняться внутри price-monitor/
%cd price-monitor

print("✅ Проект загружен")

## Шаг 3: Настройка нашей цены

Создаём файл с нашей ценой для сравнения с конкурентами. Измени число на актуальную цену.

In [ ]:
# ============================================================
# ШАГ 3: Установка нашей цены
# ============================================================

# json — встроенный модуль Python для работы с JSON-файлами
import json
import os  # os — модуль для работы с файловой системой (создание папок и т.д.)

# Наша цена в рублях за 120 г икры горбуши
# ПОМЕНЯЙ ЭТО ЧИСЛО на свою актуальную цену!
our_price = {
    "price_rub": 1590  # <-- цена за 1 банку 120 г
}

# Создаём папку data/, если её ещё нет
# exist_ok=True — значит «не ругаться, если папка уже существует»
os.makedirs("data", exist_ok=True)

# Записываем цену в JSON-файл, который читает скрипт мониторинга
# data/user_price.json — стандартный путь, куда скрипт смотрит при запуске
with open("data/user_price.json", "w", encoding="utf-8") as f:
    json.dump(our_price, f, ensure_ascii=False, indent=2)

print(f"✅ Наша цена установлена: {our_price['price_rub']:,} ₽ за 120 г".replace(",", " "))

## Шаг 4: Пояснение — как устроен сбор данных

Перед запуском посмотрим, из чего состоит проект.

In [ ]:
# ============================================================
# ШАГ 4: Обзор структуры проекта
# ============================================================

# Показываем, какие файлы есть в проекте
# Это просто для понимания — можно пропустить

print("📁 Структура проекта:\n")
print("monitor/")
print("  ├── cli.py              ← точка входа: --run --report --status")
print("  ├── tracker.py           ← ЯДРО V2: управление products.csv")
print("  ├── config.py            ← настройки: сайты, товары, цены")
print("  ├── normalize.py         ← извлечение веса, фильтрация")
print("  └── adapters/            ← сборщики для каждого сайта")
print("       ├── apeti.py        ← apeti.ru (парсинг HTML)")
print("       ├── seafood_shop.py ← seafood-shop.ru (JSON API)")
print("       └── delikateska.py  ← delikateska.ru (GraphQL API)")
print("")
print("data/")
print("  ├── products.csv         ← ОСНОВНАЯ ТАБЛИЦА (редактируешь ты!)")
print("  ├── products_YYYY-MM-DD.csv ← бэкапы с датами")
print("  └── user_price.json      ← наша цена (создали выше)")

print("\n💡 Алгоритм V2 (три прохода):")
print("  ПРОХОД 1: Собрать все продукты → найти по URL в таблице")
print("     - is_comparable='да' → обновить цену, in_stock='да'")
print("     - is_comparable='нет' → пропустить (не трогать)")
print("     - новый продукт → добавить с пустым is_comparable")
print("  ПРОХОД 2: Для 'да', не найденных в проходе 1")
print("     - Перейти по URL → проверить доступность")
print("     - Если жив → обновить цену")
print("     - Если нет → in_stock='нет'")
print("  ПРОХОД 3: Сохранить CSV + бэкап + показать отчёт")

## Шаг 5: Запуск сбора цен

Основной шаг — запускаем сбор со всех трёх сайтов и получаем таблицу.

In [ ]:
# ============================================================
# ШАГ 5: ЗАПУСК СБОРА ЦЕН (основной шаг)
# ============================================================

# python -m monitor.cli --run запускает полный цикл:
#
# ПРОХОД 1 — сбор и сопоставление:
#   1. Загружает data/products.csv (если уже есть)
#   2. Собирает ВСЕ продукты красной икры с 3 сайтов
#   3. Для каждого продукта ищет совпадение по URL в таблице:
#      - Найден и is_comparable='да'  → обновляет цену
#      - Найден и is_comparable='нет' → пропускает (не трогает)
#      - Не найден → добавляет новую строку
#
# ПРОХОД 2 — проверка отсутствующих:
#   4. Для сопоставимых продуктов, не найденных в проходе 1:
#      - Переходит по URL и проверяет, доступен ли продукт
#
# ПРОХОД 3 — сохранение:
#   5. Сохраняет обновлённый data/products.csv
#   6. Создаёт бэкап data/products_YYYY-MM-DD.csv

!python -m monitor.cli --run

## Шаг 6: Сохранённые файлы

Проверим, какие отчёты сохранились после запуска.

In [ ]:
# ============================================================
# ШАГ 6: Просмотр сохранённой таблицы
# ============================================================

import os
import csv

# Основной файл — data/products.csv
# Это твоя главная таблица со ВСЕМИ продуктами
csv_path = "data/products.csv"
if os.path.exists(csv_path):
    with open(csv_path, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    
    yes = sum(1 for r in rows if r.get('is_comparable') == 'да')
    no = sum(1 for r in rows if r.get('is_comparable') == 'нет')
    new = sum(1 for r in rows if r.get('is_comparable', '') == '')
    in_stock = sum(1 for r in rows if r.get('in_stock') == 'да')
    
    print(f"📋 data/products.csv: {len(rows)} продуктов всего")
    print(f"   Сопоставимых (да): {yes} (в наличии: {in_stock})")
    print(f"   Несопоставимых (нет): {no}")
    print(f"   Новых (не оценено): {new}")
    print()
    print("💡 Чтобы скачать CSV из Colab:")
    print("   Слева в меню открой папку data/ → правый клик на products.csv → Скачать")
    print("   Открой в Excel, поставь 'да' или 'нет' в колонке is_comparable")
    print("   При следующем запуске загрузи обновлённый CSV обратно в data/")

# Показываем бэкапы
print("\n📁 Бэкапы:")
for f in sorted(os.listdir("data")):
    if f.startswith("products_20"):
        print(f"   data/{f}")

## Шаг 7: Быстрая сводка

Показывает количество продуктов и статистику без повторного сбора данных.

In [ ]:
# ============================================================
# ШАГ 7: Быстрая сводка
# ============================================================

# --status показывает краткую статистику без повторного сбора
!python -m monitor.cli --status

---

## 🎯 Итог

**Что ты получил:**
- `data/products.csv` — таблица со ВСЕМИ продуктами красной икры с 3 сайтов
- Таблицу сравнения сопоставимых продуктов (is_comparable='да')
- Бэкап таблицы с датой (data/products_YYYY-MM-DD.csv)
- Цены, приведённые к нашему объёму (120 г) в колонке comparable_price_rub

**Твой workflow:**
1. **Первый запуск** → получил 40-50 продуктов в products.csv
2. **Открыл CSV в Excel** → поставил «да» в is_comparable для горбуши ~120 г, «нет» для остального
3. **Повторный запуск через неделю** → скрипт обновил цены для «да», пропустил «нет», добавил новинки
4. **Открыл CSV** → оценил новинки, проверил изменения цен

**Важно:**
- Всегда сохраняй `data/products.csv` между запусками — это твоя история!
- Не удаляй строки — просто ставь «нет» в is_comparable
- При каждом запуске создаётся бэкап с датой

**Где взять код:** https://github.com/tswtim/price-monitor